# CSM-SAM: Cross-Session Memory SAM
## Adaptive Radiotherapy Mid-Treatment Tumor Segmentation

**Target venue:** NeurIPS 2026  
**Benchmark:** HNTS-MRG 2024 (Head & Neck Tumor Segmentation for MR-Guided RT)  
**Key idea:** Extend SAM2's memory bank from within-scan slice propagation → **cross-visit session propagation**  
**Gap:** MedSAM2 only uses within-session memory. No prior work uses SAM2 memory across different patient imaging visits.

---

### Notebook Contents
1. 🔧 Environment Setup
2. 📥 Data Download (HNTS-MRG 2024 or Synthetic)
3. 🔬 Data Exploration & Visualization
4. 🏗️ Architecture Walkthrough
5. 🏋️ Training
6. 📊 Evaluation & Comparison Table
7. 🖼️ Visualization of Predictions
8. 📉 Ablation Studies

> **Runtime:** Use GPU (Runtime → Change runtime type → T4 GPU)  
> **Memory:** A100 recommended for full training. T4 works with batch_size=2.

## 1. 🔧 Environment Setup

In [ ]:
# Check GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU detected — use Runtime > Change runtime type > GPU')

In [ ]:
# Install core dependencies
# (PyTorch is pre-installed in Colab — we just add the extras)
import sys

!pip install -q \
    einops==0.7.0 \
    omegaconf==2.3.0 \
    SimpleITK==2.3.1 \
    nibabel==5.2.0 \
    monai==1.3.1 \
    medpy==0.5.1 \
    timm==0.9.16 \
    wandb==0.16.4 \
    zenodo-get==1.3.4 \
    surface-distance==0.1

print('Core dependencies installed.')

In [ ]:
import os, sys

# Install SAM2 from Meta
if not os.path.exists("sam2"):
    !git clone -q https://github.com/facebookresearch/sam2.git
    !pip install -q -e sam2/
    print("SAM2 installed.")
else:
    print("SAM2 already cloned.")

# Install CSM-SAM (this repo)
if not os.path.exists("csm-sam"):
    !git clone -q https://github.com/NatalieCarlebach1/csm-sam.git
    !pip install -q -e csm-sam/
    print("CSM-SAM installed.")
else:
    # Pull latest changes
    !git -C csm-sam pull -q
    print("CSM-SAM already cloned (pulled latest).")

# Add repo root to Python path so all subpackages resolve
repo_root = os.path.abspath("csm-sam")
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
if os.path.abspath("sam2") not in sys.path:
    sys.path.insert(0, os.path.abspath("sam2"))

print(f"Repo root on path: {repo_root}")


In [ ]:
# Download SAM2.1 Hiera-Large checkpoint (~900MB)
import os
os.makedirs('checkpoints/sam2', exist_ok=True)

checkpoint_path = 'checkpoints/sam2/sam2.1_hiera_large.pt'
if not os.path.exists(checkpoint_path):
    print('Downloading SAM2.1 checkpoint (~900MB)...')
    !wget -q --show-progress \
        -O {checkpoint_path} \
        'https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt'
    print('Checkpoint downloaded.')
else:
    print(f'Checkpoint already exists: {checkpoint_path}')

In [ ]:
# Standard imports
import torch
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. 📥 Data Download

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# CHOOSE DATA SOURCE:
#
#  Option A: SYNTHETIC (default) — runs immediately, no registration needed
#            Use for testing the pipeline end-to-end.
#            NOT suitable for real experiments.
#
#  Option B: REAL HNTS-MRG 2024 data
#            Requires registration at https://hnts-mrg.grand-challenge.org/
#            Then: zenodo_get 11829006 -o data/raw/
# ─────────────────────────────────────────────────────────────────────

USE_SYNTHETIC = True   # ← Set to False when you have real data
N_SYNTHETIC   = 20     # Number of synthetic patients (train+val+test)

DATA_DIR = Path('data/processed')
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f'Data mode: {"SYNTHETIC" if USE_SYNTHETIC else "REAL"}')
print(f'Data directory: {DATA_DIR}')

In [ ]:
if USE_SYNTHETIC:
    import sys, importlib, os
    # Ensure the repo root is on the path
    repo_root = os.path.abspath("csm-sam")
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)

    # Import using importlib to bypass package detection issues
    import importlib.util
    spec = importlib.util.spec_from_file_location(
        "download_hnts_mrg",
        os.path.join(repo_root, "data", "download_hnts_mrg.py")
    )
    dl_mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(dl_mod)
    create_synthetic_dataset = dl_mod.create_synthetic_dataset

    create_synthetic_dataset(
        output_dir=DATA_DIR,
        n_patients=N_SYNTHETIC,
        n_slices=32,
    )
    print(f"
Synthetic dataset ready at {DATA_DIR}")

    # Verify structure
    for split in ["train", "val", "test"]:
        patients = list((DATA_DIR / split).iterdir())
        print(f"  {split}: {len(patients)} patients")

else:
    # Real data: run after registering and downloading
    !python csm-sam/data/preprocess.py \n        --input_dir data/raw \n        --output_dir data/processed \n        --n_workers 2


## 3. 🔬 Data Exploration

In [ ]:
import SimpleITK as sitk

# Pick a random training patient
train_dir = DATA_DIR / 'train'
patients = sorted(train_dir.iterdir())
patient = patients[0]
print(f'Patient: {patient.name}')

# Load volumes
def load_nii(path):
    img = sitk.ReadImage(str(path))
    return sitk.GetArrayFromImage(img)

pre_vol  = load_nii(patient / 'pre_image.nii.gz')
mid_vol  = load_nii(patient / 'mid_image.nii.gz')
pre_gtvp = load_nii(patient / 'pre_GTVp.nii.gz')
mid_gtvp = load_nii(patient / 'mid_GTVp.nii.gz')

print(f'Volume shape (D, H, W): {pre_vol.shape}')
print(f'Pre-RT tumor volume:    {pre_gtvp.sum():.0f} voxels')
print(f'Mid-RT tumor volume:    {mid_gtvp.sum():.0f} voxels')
shrinkage = (1 - mid_gtvp.sum() / (pre_gtvp.sum() + 1e-6)) * 100
print(f'Tumor shrinkage:        {shrinkage:.1f}%  ← treatment response')

In [ ]:
# Visualize pre-RT vs mid-RT with tumor masks
# Find the best slice (largest tumor)
best_slice = int(np.argmax([pre_gtvp[i].sum() + mid_gtvp[i].sum() for i in range(pre_vol.shape[0])]))

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
fig.suptitle(f'Patient: {patient.name} | Best slice: z={best_slice}', fontsize=13)

def overlay(img_slice, mask_slice, color=(1, 0.2, 0.2), alpha=0.45):
    rgb = np.stack([img_slice]*3, axis=-1).copy()
    mask_bool = mask_slice.astype(bool)
    for c, v in enumerate(color):
        rgb[:, :, c][mask_bool] = (1 - alpha) * rgb[:, :, c][mask_bool] + alpha * v
    return rgb.clip(0, 1)

for row, (vol, gtvp_vol, label) in enumerate([
    (pre_vol, pre_gtvp, 'Pre-RT'),
    (mid_vol, mid_gtvp, 'Mid-RT'),
]):
    slices_to_show = [max(0, best_slice - 4), max(0, best_slice - 2), best_slice, min(vol.shape[0]-1, best_slice + 2)]
    for col, sl in enumerate(slices_to_show):
        img_sl = vol[sl].clip(0, 1)
        mask_sl = gtvp_vol[sl] > 0.5
        axes[row, col].imshow(overlay(img_sl, mask_sl), cmap='gray', vmin=0, vmax=1)
        axes[row, col].set_title(f'{label} z={sl}', fontsize=9)
        axes[row, col].axis('off')

plt.tight_layout()
plt.savefig('data_exploration.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: data_exploration.png')

In [ ]:
# Visualize the change map (what our ChangeHead predicts)
pre_mask = (pre_gtvp[best_slice] > 0.5).astype(int)
mid_mask = (mid_gtvp[best_slice] > 0.5).astype(int)

change_map = np.zeros_like(pre_mask)
change_map[(pre_mask == 1) & (mid_mask == 1)] = 1  # stable
change_map[(pre_mask == 0) & (mid_mask == 1)] = 2  # grown
change_map[(pre_mask == 1) & (mid_mask == 0)] = 3  # shrunk

CHANGE_RGB = {
    0: [30,  30,  30],   # background: dark
    1: [255, 220,  0],   # stable: yellow
    2: [255,  50, 50],   # grown: red
    3: [50,  120, 255],  # shrunk: blue
}

change_rgb = np.zeros((*change_map.shape, 3), dtype=np.uint8)
for cls, color in CHANGE_RGB.items():
    change_rgb[change_map == cls] = color

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
fig.suptitle('Automatic Change Map Labels (no extra annotation needed)', fontsize=12)

axes[0].imshow(pre_vol[best_slice], cmap='gray'); axes[0].set_title('Pre-RT MRI'); axes[0].axis('off')
axes[1].imshow(mid_vol[best_slice], cmap='gray'); axes[1].set_title('Mid-RT MRI'); axes[1].axis('off')

axes[2].imshow(pre_vol[best_slice], cmap='gray')
axes[2].imshow(np.where(pre_mask[:,:,None], [[200,50,50]], 0).astype(np.uint8), alpha=0.4)
axes[2].imshow(np.where(mid_mask[:,:,None], [[50,200,50]], 0).astype(np.uint8), alpha=0.4)
axes[2].set_title('Pre(red) + Mid(green)'); axes[2].axis('off')

axes[3].imshow(change_rgb)
axes[3].set_title('Change Map')
axes[3].axis('off')

patches = [
    mpatches.Patch(color=np.array(CHANGE_RGB[1])/255, label='Stable tumor'),
    mpatches.Patch(color=np.array(CHANGE_RGB[2])/255, label='Grown region'),
    mpatches.Patch(color=np.array(CHANGE_RGB[3])/255, label='Shrunk region'),
]
fig.legend(handles=patches, loc='lower center', ncol=3, fontsize=10, bbox_to_anchor=(0.5, -0.05))
plt.tight_layout()
plt.savefig('change_map_labels.png', dpi=130, bbox_inches='tight')
plt.show()

## 4. 🏗️ Architecture Walkthrough

In [ ]:
import torch
import torch.nn as nn
import sys
sys.path.insert(0, 'csm-sam')

from csmsam.modeling.cross_session_memory_attention import (
    CrossSessionMemoryAttention,
    CrossSessionMemoryEncoder,
    TemporalEmbedding,
)
from csmsam.modeling.change_head import ChangeHead, build_change_labels
from csmsam.losses.combined_loss import CombinedLoss

print('All modules imported successfully.')
print()
print('=== Architecture Overview ===')
print()
print('Pre-RT scan  ──► SAM2 Encoder (frozen) ──► CrossSessionMemoryEncoder')
print('                                                      │')
print('                                                   M_pre (memory bank)')
print('                                                      │')
print('Mid-RT scan  ──► SAM2 Encoder (frozen) ──► CrossSessionMemoryAttention ◄── weeks_elapsed')
print('                         │                          │')
print('                      M_mid ───────────────────────►│')
print('                                                      │')
print('                                               Mask Decoder ──► Segmentation')
print('                                                      │')
print('                                              ChangeHead ──► Change Map (stable/grown/shrunk)')

In [ ]:
# ─── Test CrossSessionMemoryAttention in isolation ───
D_MODEL = 256
B = 2       # batch size
HW = 64*64  # spatial tokens (64×64 feature map = 1024 image / 16 patch size)
N_PRE = 8 * 16 * 16  # 8 pre-RT slices, each pooled to 16×16
N_MID = 2 * 64 * 64  # 2 prior mid-RT slices

# Inputs
curr_features  = torch.randn(B, HW, D_MODEL)
M_pre          = torch.randn(B, N_PRE, D_MODEL)
M_mid          = torch.randn(B, N_MID, D_MODEL)
weeks_elapsed  = torch.tensor([3, 2], dtype=torch.long)  # patient 0: week 3, patient 1: week 2

# Module
csma = CrossSessionMemoryAttention(d_model=D_MODEL, num_heads=8)

with torch.no_grad():
    out, attn_w, gate_vals = csma(
        curr_features=curr_features,
        M_pre=M_pre,
        M_mid=M_mid,
        weeks_elapsed=weeks_elapsed,
    )

print('CrossSessionMemoryAttention:')
print(f'  Input:      curr_features  {curr_features.shape}')
print(f'  Memory:     M_pre          {M_pre.shape}')
print(f'              M_mid          {M_mid.shape}')
print(f'  Output:     enhanced_feat  {out.shape}')
print(f'  Attn wts:   attn_weights   {attn_w.shape}')
print(f'  Gate vals:  gate           {gate_vals.shape}')
print(f'  Gate mean: {gate_vals.mean():.3f} (0=within-session, 1=cross-session)')

# Count parameters
n_params = sum(p.numel() for p in csma.parameters() if p.requires_grad)
print(f'  Trainable params: {n_params:,}')

In [ ]:
# ─── Test ChangeHead ───
C, H_feat, W_feat = 256, 64, 64

pre_features = torch.randn(B, C, H_feat, W_feat)
mid_features = torch.randn(B, C, H_feat, W_feat)

change_head = ChangeHead(in_channels=C, num_classes=4)

with torch.no_grad():
    change_logits = change_head(pre_features, mid_features)

print('ChangeHead:')
print(f'  Input:  pre_features  {pre_features.shape}')
print(f'          mid_features  {mid_features.shape}')
print(f'  Output: change_logits {change_logits.shape}  ← 4 classes: bg/stable/grown/shrunk')

# Build automatic change labels
pre_mask_t = torch.randint(0, 2, (B, 1, H_feat, W_feat)).float()
mid_mask_t = torch.randint(0, 2, (B, 1, H_feat, W_feat)).float()
change_labels = build_change_labels(pre_mask_t, mid_mask_t)

class_counts = [(change_labels == c).sum().item() for c in range(4)]
print(f'\nChange label distribution (from mask XOR):')
for cls, name, cnt in zip(range(4), ['background', 'stable', 'grown', 'shrunk'], class_counts):
    print(f'  class {cls} ({name}): {cnt:5d} pixels')

n_params = sum(p.numel() for p in change_head.parameters() if p.requires_grad)
print(f'\nChangeHead trainable params: {n_params:,}')

In [ ]:
# ─── Test CombinedLoss ───
from csmsam.losses.combined_loss import CombinedLoss

loss_fn = CombinedLoss(lambda_dice=1.0, lambda_bce=1.0, lambda_change=0.3)

pred_masks   = torch.randn(B, 1, 256, 256)
target_masks = torch.randint(0, 2, (B, 1, 256, 256)).float()
pre_masks_l  = torch.randint(0, 2, (B, 1, 256, 256)).float()
change_logits_l = torch.randn(B, 4, 256, 256)

losses = loss_fn(
    pred_masks=pred_masks,
    target_masks=target_masks,
    change_logits=change_logits_l,
    pre_masks=pre_masks_l,
)

print('CombinedLoss:')
print(f"  Dice loss:   {losses['dice']:.4f}")
print(f"  BCE loss:    {losses['bce']:.4f}")
print(f"  Change loss: {losses['change']:.4f}")
print(f"  Total loss:  {losses['total']:.4f}")

In [ ]:
# ─── Total parameter count ───
from csmsam.modeling.cross_session_memory_attention import CrossSessionMemoryEncoder

mem_encoder = CrossSessionMemoryEncoder(d_model=256, n_memory_frames=8, spatial_pool_size=16)
cross_attn  = CrossSessionMemoryAttention(d_model=256, num_heads=8)
change_head = ChangeHead(in_channels=256, num_classes=4)

components = {
    'CrossSessionMemoryEncoder': mem_encoder,
    'CrossSessionMemoryAttention': cross_attn,
    'ChangeHead': change_head,
}

print('Trainable Parameters per Component:')
print('-' * 45)
total = 0
for name, module in components.items():
    n = sum(p.numel() for p in module.parameters() if p.requires_grad)
    total += n
    print(f'  {name:<35} {n:>8,}')
print('-' * 45)
print(f'  {"TOTAL (novel modules)":<35} {total:>8,}')
print()
print('  SAM2-L encoder (frozen):              ~307,000,000')
print('  SAM2-L decoder (frozen, except head):  ~6,400,000')
print(f'  CSM-SAM novel modules (trained):        {total:>10,}')
print(f'  Training efficiency: {total/307e6*100:.2f}% of SAM2-L encoder')

## 5. 🏋️ Training

In [ ]:
from csmsam.datasets import HNTSMRGSliceDataset, HNTSMRGDataset
from torch.utils.data import DataLoader

# Use smaller image size for Colab (512 instead of 1024 to fit in memory)
IMAGE_SIZE = 512   # Use 1024 for full experiments

train_ds = HNTSMRGSliceDataset(
    data_dir=str(DATA_DIR),
    split='train',
    image_size=IMAGE_SIZE,
    augment=True,
)
val_ds = HNTSMRGDataset(
    data_dir=str(DATA_DIR),
    split='val',
    image_size=IMAGE_SIZE,
)

train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=1, shuffle=False, num_workers=1)

print(f'Train dataset: {len(train_ds)} slices  ({len(train_loader)} batches)')
print(f'Val dataset:   {len(val_ds)} patients  ({len(val_loader)} batches)')

# Show one training batch
batch = next(iter(train_loader))
print(f'\nBatch keys: {list(batch.keys())}')
print(f'pre_image:    {batch["pre_image"].shape}')
print(f'mid_image:    {batch["mid_image"].shape}')
print(f'mid_mask:     {batch["mid_mask"].shape}')
print(f'weeks_elapsed:{batch["weeks_elapsed"]}')

In [ ]:
# Build a lightweight CSM-SAM for Colab (without full SAM2 — fallback decoder)
# For full training, install SAM2 and use CSMSAM.from_pretrained()

import torch
import torch.nn as nn
import torch.nn.functional as F

from csmsam.modeling.cross_session_memory_attention import (
    CrossSessionMemoryAttention,
    CrossSessionMemoryEncoder,
)
from csmsam.modeling.change_head import ChangeHead
from csmsam.losses.combined_loss import CombinedLoss


class ColabCSMSAM(nn.Module):
    """
    Lightweight CSM-SAM for Colab demo without full SAM2 installation.
    Replaces SAM2 encoder with a simple CNN feature extractor.
    Same novel modules (CSMA, ChangeHead) — only the backbone changes.
    """

    def __init__(self, d_model=128, image_size=512):
        super().__init__()
        self.d_model = d_model
        self.feat_size = image_size // 16  # simulating ViT patch size 16

        # Lightweight CNN encoder (replaces SAM2 ViT-H)
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, d_model, 3, stride=2, padding=1), nn.BatchNorm2d(d_model), nn.ReLU(),
            nn.Conv2d(d_model, d_model, 3, stride=2, padding=1), nn.BatchNorm2d(d_model), nn.ReLU(),
        )  # output: (B, d_model, H/16, W/16)

        # Novel CSM-SAM modules
        self.memory_encoder = CrossSessionMemoryEncoder(
            d_model=d_model, n_memory_frames=4, spatial_pool_size=8
        )
        self.cross_session_attn = CrossSessionMemoryAttention(
            d_model=d_model, num_heads=4, max_weeks=12
        )
        self.change_head = ChangeHead(in_channels=d_model, num_classes=4)

        # Simple mask decoder
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(d_model, 64, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 16, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(16, 1, 4, stride=2, padding=1),
        )  # output: (B, 1, H, W)

        self._M_mid = None

    def get_features(self, images):
        return self.encoder(images)  # (B, d, h, w)

    def encode_pre_rt(self, pre_images, pre_masks=None):
        B, N, C, H, W = pre_images.shape
        feats = torch.stack([self.get_features(pre_images[:, i]) for i in range(N)], dim=1)
        return self.memory_encoder(feats)  # (B, N_mem, d)

    def reset_mid_session_memory(self):
        self._M_mid = None

    def forward(self, mid_images, M_pre, pre_images=None, weeks_elapsed=None, return_change_map=True):
        B, _, H, W = mid_images.shape
        if weeks_elapsed is None:
            weeks_elapsed = torch.full((B,), 3, dtype=torch.long, device=mid_images.device)

        mid_feats = self.get_features(mid_images)  # (B, d, h, w)
        h, w = mid_feats.shape[-2:]
        d = mid_feats.shape[1]

        mid_flat = mid_feats.permute(0, 2, 3, 1).reshape(B, h*w, d)

        enhanced, attn_w, gate_vals = self.cross_session_attn(
            curr_features=mid_flat,
            M_pre=M_pre,
            M_mid=self._M_mid,
            weeks_elapsed=weeks_elapsed,
        )

        self._M_mid = enhanced.detach()

        enhanced_spatial = enhanced.reshape(B, h, w, d).permute(0, 3, 1, 2)
        masks = self.decoder(enhanced_spatial)
        masks = F.interpolate(masks, size=(H, W), mode='bilinear', align_corners=False)

        result = {
            'masks': masks,
            'gate_vals': gate_vals.reshape(B, h, w),
            'attn_weights': attn_w.mean(dim=1),
        }

        if return_change_map and pre_images is not None:
            pre_feats = self.get_features(pre_images)
            change_logits = self.change_head(pre_feats, enhanced_spatial)
            change_logits = F.interpolate(change_logits, size=(H, W), mode='bilinear', align_corners=False)
            result['change_map'] = change_logits

        return result


model = ColabCSMSAM(d_model=128, image_size=IMAGE_SIZE).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'ColabCSMSAM model built. Trainable params: {n_params:,}')
print('(In full experiments, replace with CSMSAM.from_pretrained() using SAM2-L backbone)')

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import GradScaler, autocast
import time

EPOCHS   = 30    # Increase to 200 for real training
LR       = 1e-3
USE_AMP  = (DEVICE == 'cuda')

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
loss_fn   = CombinedLoss(lambda_dice=1.0, lambda_bce=1.0, lambda_change=0.3)
scaler    = GradScaler(enabled=USE_AMP)

train_history = []

print(f'Training for {EPOCHS} epochs on {DEVICE}...')
print(f'Steps per epoch: {len(train_loader)}')
print()

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_losses = []
    t0 = time.time()

    for batch in train_loader:
        mid_images = batch['mid_image'].to(DEVICE)
        pre_images = batch['pre_image'].to(DEVICE)
        mid_masks  = batch['mid_mask'].to(DEVICE)
        pre_masks  = batch['pre_mask'].to(DEVICE)
        weeks = batch['weeks_elapsed']
        if isinstance(weeks, list):
            weeks = torch.tensor(weeks, dtype=torch.long)
        weeks = weeks.to(DEVICE)

        with autocast(enabled=USE_AMP):
            M_pre = model.encode_pre_rt(pre_images.unsqueeze(1))
            model.reset_mid_session_memory()
            out = model(
                mid_images=mid_images,
                M_pre=M_pre,
                pre_images=pre_images,
                weeks_elapsed=weeks,
            )
            losses = loss_fn(
                pred_masks=out['masks'],
                target_masks=mid_masks,
                change_logits=out.get('change_map'),
                pre_masks=pre_masks,
            )

        optimizer.zero_grad()
        scaler.scale(losses['total']).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        epoch_losses.append({
            'total': losses['total'].item(),
            'dice':  losses.get('dice', torch.tensor(0)).item(),
        })

    scheduler.step()

    avg_total = np.mean([l['total'] for l in epoch_losses])
    avg_dice  = np.mean([l['dice']  for l in epoch_losses])
    elapsed   = time.time() - t0

    train_history.append({'epoch': epoch, 'total': avg_total, 'dice': avg_dice})

    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{EPOCHS} | loss={avg_total:.4f} | dice_loss={avg_dice:.4f} | {elapsed:.1f}s')

print('\nTraining complete!')

In [ ]:
# Plot training curves
epochs = [h['epoch'] for h in train_history]
totals = [h['total'] for h in train_history]
dices  = [h['dice']  for h in train_history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(epochs, totals, 'b-', linewidth=2, label='Total loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('Total Training Loss')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(epochs, dices, 'r-', linewidth=2, label='Dice loss')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss'); ax2.set_title('Dice Loss (lower = better segmentation)')
ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle('CSM-SAM Training Curves', fontsize=12)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=130, bbox_inches='tight')
plt.show()
print(f'Final total loss: {totals[-1]:.4f}')
print(f'Final dice loss:  {dices[-1]:.4f}')

In [ ]:
# Save checkpoint
import os
os.makedirs('checkpoints/colab', exist_ok=True)
torch.save({'model_state_dict': model.state_dict(), 'history': train_history}, 'checkpoints/colab/csmsam_colab.pth')
print('Checkpoint saved: checkpoints/colab/csmsam_colab.pth')

## 6. 📊 Evaluation & Metrics

In [ ]:
from csmsam.utils.metrics import compute_dice, compute_agg_dsc

model.eval()
test_ds = HNTSMRGDataset(data_dir=str(DATA_DIR), split='test', image_size=IMAGE_SIZE)

all_metrics = []

with torch.no_grad():
    for patient_data in test_ds:
        pre_images = patient_data['pre_images'].to(DEVICE)  # (N, 3, H, W)
        mid_images = patient_data['mid_images'].to(DEVICE)
        mid_masks_gt = patient_data['mid_masks'].squeeze(1).numpy()  # (N, H, W)
        weeks = torch.tensor([int(patient_data['weeks_elapsed'])], dtype=torch.long, device=DEVICE)

        M_pre = model.encode_pre_rt(pre_images.unsqueeze(0))
        model.reset_mid_session_memory()

        pred_slices = []
        for i in range(pre_images.shape[0]):
            out = model(
                mid_images=mid_images[i].unsqueeze(0),
                M_pre=M_pre,
                pre_images=pre_images[i].unsqueeze(0),
                weeks_elapsed=weeks,
                return_change_map=False,
            )
            pred = (torch.sigmoid(out['masks']) > 0.5).squeeze().cpu().numpy()
            pred_slices.append(pred)

        pred_vol = np.stack(pred_slices)
        dsc = compute_dice(pred_vol, mid_masks_gt > 0.5)
        all_metrics.append({'patient_id': patient_data['patient_id'], 'dsc': dsc})

dsc_values = [m['dsc'] for m in all_metrics]
print('=' * 40)
print('Test Set Evaluation')
print('=' * 40)
print(f'Mean DSC:   {np.mean(dsc_values):.4f}')
print(f'Median DSC: {np.median(dsc_values):.4f}')
print(f'Std DSC:    {np.std(dsc_values):.4f}')
print(f'Min / Max:  {np.min(dsc_values):.4f} / {np.max(dsc_values):.4f}')
print()
print('Per-patient:')
for m in sorted(all_metrics, key=lambda x: x['dsc'], reverse=True):
    bar = '█' * int(m['dsc'] * 20)
    print(f"  {m['patient_id']:<25} DSC={m['dsc']:.3f} {bar}")

In [ ]:
# Comparison table against published baselines
our_dsc = float(np.mean(dsc_values))

results = [
    ('HNTS-MRG 2024 Winner†',        0.727, '—',    '—',    '—'),
    ('HNTS-MRG 2024 Challenge Mean†', 0.688, '—',    '—',    '—'),
    ('MedSAM2 (within-session only)', None,  '—',    '—',    '—'),  # run baseline to fill
    ('CSM-SAM (Colab demo)',          our_dsc, '—',  '—',    '—'),
    ('CSM-SAM (Full, target)',        '>0.75', '—',  '—',    '—'),
]

print(f"{'Method':<35} {'aggDSC':>8} {'GTVp DSC':>10} {'GTVn DSC':>10} {'HD95':>8}")
print('-' * 75)
for method, agg, gtvp, gtvn, hd in results:
    agg_str = f'{agg:.4f}' if isinstance(agg, float) else str(agg)
    print(f'{method:<35} {agg_str:>8} {str(gtvp):>10} {str(gtvn):>10} {str(hd):>8}')
print('-' * 75)
print('†Published results from HNTS-MRG 2024 challenge (Wahid et al., 2024)')
print('Note: Colab demo uses simplified CNN encoder, not SAM2. Use train.py for real results.')

## 7. 🖼️ Visualization of Predictions

In [ ]:
import random

# Pick 3 random test patients and visualize
N_VIS = min(3, len(test_ds))
random.seed(42)
vis_indices = random.sample(range(len(test_ds)), N_VIS)

model.eval()

for patient_idx in vis_indices:
    patient_data = test_ds[patient_idx]
    pid = patient_data['patient_id']

    pre_images = patient_data['pre_images'].to(DEVICE)
    mid_images = patient_data['mid_images'].to(DEVICE)
    mid_masks_gt = patient_data['mid_masks']  # (N, 1, H, W)
    weeks = torch.tensor([int(patient_data['weeks_elapsed'])], dtype=torch.long, device=DEVICE)

    with torch.no_grad():
        M_pre = model.encode_pre_rt(pre_images.unsqueeze(0))
        model.reset_mid_session_memory()

        pred_masks, change_maps, gate_vals = [], [], []
        for i in range(pre_images.shape[0]):
            out = model(
                mid_images=mid_images[i].unsqueeze(0),
                M_pre=M_pre,
                pre_images=pre_images[i].unsqueeze(0),
                weeks_elapsed=weeks,
                return_change_map=True,
            )
            pred_masks.append(out['masks'].squeeze().cpu())
            if 'change_map' in out:
                change_maps.append(out['change_map'].squeeze().cpu())
            gate_vals.append(out['gate_vals'].squeeze().cpu())

    # Find best slice (largest GT tumor)
    tumor_sizes = [mid_masks_gt[i].sum().item() for i in range(len(mid_masks_gt))]
    best_sl = int(np.argmax(tumor_sizes)) if max(tumor_sizes) > 0 else len(pred_masks) // 2

    # De-normalize image for display
    SAM2_MEAN = np.array([0.485, 0.456, 0.406])
    SAM2_STD  = np.array([0.229, 0.224, 0.225])

    def denorm(t):
        arr = t.permute(1,2,0).numpy()
        return (arr * SAM2_STD + SAM2_MEAN).clip(0, 1)

    pre_img = denorm(patient_data['pre_images'][best_sl].cpu())
    mid_img = denorm(patient_data['mid_images'][best_sl].cpu())

    gt = mid_masks_gt[best_sl].squeeze().numpy() > 0.5
    pred = (torch.sigmoid(pred_masks[best_sl]) > 0.5).numpy()
    gate = gate_vals[best_sl].numpy() if gate_vals else None

    # Dice for this slice
    inter = (pred & gt).sum()
    dice_sl = (2 * inter + 1e-6) / (pred.sum() + gt.sum() + 1e-6)

    n_cols = 5 if gate is not None else 4
    fig, axes = plt.subplots(1, n_cols, figsize=(n_cols * 3.8, 4))
    fig.suptitle(f'{pid} | Slice z={best_sl} | DSC={dice_sl:.3f} | Week {int(weeks[0])}', fontsize=11)

    def ov(img, mask, color=(1, 0.2, 0.2), alpha=0.45):
        out = img.copy()
        out[mask, 0] = (1-alpha)*img[mask,0] + alpha*color[0]
        out[mask, 1] = (1-alpha)*img[mask,1] + alpha*color[1]
        out[mask, 2] = (1-alpha)*img[mask,2] + alpha*color[2]
        return out

    axes[0].imshow(pre_img); axes[0].set_title('Pre-RT MRI'); axes[0].axis('off')
    axes[1].imshow(ov(mid_img, gt, (1,0.2,0.2))); axes[1].set_title('Mid-RT + GT'); axes[1].axis('off')
    axes[2].imshow(ov(mid_img, pred, (0.2,1,0.2))); axes[2].set_title(f'Prediction (DSC={dice_sl:.3f})'); axes[2].axis('off')

    # GT vs Pred overlap
    overlap_img = mid_img.copy() * 0.4
    overlap_img[gt,  0] += 0.6
    overlap_img[pred,1] += 0.6
    overlap_img[gt & pred, :] = [0.8, 0.8, 0.0]
    axes[3].imshow(overlap_img.clip(0,1))
    axes[3].set_title('GT(red) Pred(green) Overlap(yellow)')
    axes[3].axis('off')

    if gate is not None:
        im = axes[4].imshow(gate, cmap='plasma', vmin=0, vmax=1)
        plt.colorbar(im, ax=axes[4], fraction=0.046)
        axes[4].set_title('Cross-session gate\n(1=pre-RT memory)')
        axes[4].axis('off')

    plt.tight_layout()
    plt.savefig(f'prediction_{pid}.png', dpi=130, bbox_inches='tight')
    plt.show()

print('Visualization complete.')

In [ ]:
# Slice gallery for the first test patient
patient_data = test_ds[0]
pid = patient_data['patient_id']

pre_images = patient_data['pre_images'].to(DEVICE)
mid_images = patient_data['mid_images'].to(DEVICE)
weeks = torch.tensor([int(patient_data['weeks_elapsed'])], dtype=torch.long, device=DEVICE)

with torch.no_grad():
    M_pre = model.encode_pre_rt(pre_images.unsqueeze(0))
    model.reset_mid_session_memory()
    pred_slices_all = []
    for i in range(pre_images.shape[0]):
        out = model(mid_images[i].unsqueeze(0), M_pre, pre_images[i].unsqueeze(0), weeks, return_change_map=False)
        pred_slices_all.append((torch.sigmoid(out['masks']) > 0.5).squeeze().cpu().numpy())

N = len(pred_slices_all)
n_show = min(8, N)
idx_show = np.linspace(0, N-1, n_show, dtype=int)

SAM2_MEAN = np.array([0.485, 0.456, 0.406])
SAM2_STD  = np.array([0.229, 0.224, 0.225])

def denorm(t):
    arr = t.permute(1,2,0).numpy()
    return (arr * SAM2_STD + SAM2_MEAN).clip(0, 1)

fig, axes = plt.subplots(2, n_show, figsize=(n_show * 2.5, 5.5))
fig.suptitle(f'Slice Gallery — {pid}', fontsize=12)

for col, sl in enumerate(idx_show):
    img = denorm(patient_data['mid_images'][sl])
    gt  = patient_data['mid_masks'][sl].squeeze().numpy() > 0.5
    pred = pred_slices_all[sl]

    # GT row
    gt_vis = img.copy()
    gt_vis[gt, 0] = 0.7; gt_vis[gt, 1] = 0.2; gt_vis[gt, 2] = 0.2
    axes[0, col].imshow(gt_vis.clip(0, 1))
    axes[0, col].set_title(f'GT z={sl}', fontsize=8)
    axes[0, col].axis('off')

    # Pred row
    pred_vis = img.copy()
    pred_vis[pred, 0] = 0.2; pred_vis[pred, 1] = 0.7; pred_vis[pred, 2] = 0.2
    inter = (gt & pred).sum()
    dsc_sl = (2*inter+1e-6)/(gt.sum()+pred.sum()+1e-6)
    axes[1, col].imshow(pred_vis.clip(0, 1))
    axes[1, col].set_title(f'Pred DSC={dsc_sl:.2f}', fontsize=8)
    axes[1, col].axis('off')

axes[0, 0].set_ylabel('GT', fontsize=9)
axes[1, 0].set_ylabel('Pred', fontsize=9)

plt.tight_layout()
plt.savefig(f'gallery_{pid}.png', dpi=130, bbox_inches='tight')
plt.show()

## 8. 📉 Ablation Studies

In [ ]:
# Quick ablation: CSM-SAM vs No Cross-Session (M_pre replaced with zeros)
# This directly shows the contribution of cross-session memory.

model.eval()
ablation_results = {}

configs = [
    ('CSM-SAM (full)',          True,   3),
    ('No cross-session memory', False,  3),   # ablation: M_pre = zeros
    ('Week 1 (early RT)',       True,   1),   # temporal condition variation
    ('Week 4 (late RT)',        True,   4),
]

for config_name, use_pre_memory, weeks_val in configs:
    dsc_list = []
    with torch.no_grad():
        for patient_data in test_ds:
            pre_images = patient_data['pre_images'].to(DEVICE)
            mid_images = patient_data['mid_images'].to(DEVICE)
            mid_masks_gt = patient_data['mid_masks'].squeeze(1).numpy()
            weeks = torch.tensor([weeks_val], dtype=torch.long, device=DEVICE)

            M_pre = model.encode_pre_rt(pre_images.unsqueeze(0))

            if not use_pre_memory:
                # Ablation: replace cross-session memory with zeros
                M_pre = torch.zeros_like(M_pre)

            model.reset_mid_session_memory()
            pred_slices = []
            for i in range(pre_images.shape[0]):
                out = model(mid_images[i].unsqueeze(0), M_pre, pre_images[i].unsqueeze(0), weeks, return_change_map=False)
                pred = (torch.sigmoid(out['masks']) > 0.5).squeeze().cpu().numpy()
                pred_slices.append(pred)

            pred_vol = np.stack(pred_slices)
            dsc = compute_dice(pred_vol, mid_masks_gt > 0.5)
            dsc_list.append(dsc)

    ablation_results[config_name] = np.mean(dsc_list)

print('=' * 50)
print('Ablation Study Results')
print('=' * 50)
baseline_dsc = ablation_results.get('No cross-session memory', 0)
for name, dsc in ablation_results.items():
    delta = dsc - baseline_dsc
    delta_str = f'(+{delta:.4f})' if delta > 0 else f'({delta:.4f})'
    print(f'  {name:<35} DSC={dsc:.4f}  {delta_str}')
print('=' * 50)
print('The delta shows the contribution of each component.')

In [ ]:
# Plot ablation results
names = list(ablation_results.keys())
values = [ablation_results[n] for n in names]
colors = ['steelblue' if 'full' in n.lower() else 'lightcoral' for n in names]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(range(len(names)), values, color=colors, edgecolor='white', linewidth=1.5)

for bar, val in zip(bars, values):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2, f'{val:.4f}',
            va='center', fontsize=10, fontweight='bold')

ax.set_yticks(range(len(names)))
ax.set_yticklabels(names, fontsize=10)
ax.set_xlabel('Mean DSC', fontsize=11)
ax.set_title('CSM-SAM Ablation Study\n(Higher is better)', fontsize=12)
ax.set_xlim(0, min(1.0, max(values) + 0.1))
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('ablation_results.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# Analyze gate values: does cross-session gate decrease as more within-session context accumulates?
# Expected: gate ≈ 1 at early slices (only pre-RT memory), decreases as M_mid grows

patient_data = test_ds[0]
pre_images = patient_data['pre_images'].to(DEVICE)
mid_images = patient_data['mid_images'].to(DEVICE)
weeks = torch.tensor([3], dtype=torch.long, device=DEVICE)

with torch.no_grad():
    M_pre = model.encode_pre_rt(pre_images.unsqueeze(0))
    model.reset_mid_session_memory()

    gate_means = []
    for i in range(pre_images.shape[0]):
        out = model(mid_images[i].unsqueeze(0), M_pre, pre_images[i].unsqueeze(0), weeks, return_change_map=False)
        gate = out['gate_vals'].mean().item()
        gate_means.append(gate)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(gate_means, 'b-o', markersize=4, linewidth=2, label='Mean gate value')
ax.axhline(0.5, color='red', linestyle='--', alpha=0.5, label='Gate=0.5 (balanced)')
ax.fill_between(range(len(gate_means)), gate_means, alpha=0.15)
ax.set_xlabel('Slice index (inferior → superior)', fontsize=11)
ax.set_ylabel('Cross-session gate value', fontsize=11)
ax.set_title('Cross-Session Gate vs Slice Index\n(1 = use pre-RT memory, 0 = use within-session memory)', fontsize=11)
ax.set_ylim(0, 1)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('gate_analysis.png', dpi=130, bbox_inches='tight')
plt.show()

print(f'Average gate value across all slices: {np.mean(gate_means):.3f}')
print(f'Gate at slice 0  (start): {gate_means[0]:.3f}')
print(f'Gate at slice -1 (end):   {gate_means[-1]:.3f}')

## 9. 📦 Download Results

Download all generated figures and the checkpoint.

In [ ]:
import os, zipfile

# Collect all generated files
files_to_zip = [f for f in os.listdir('.') if f.endswith('.png')]
files_to_zip.append('checkpoints/colab/csmsam_colab.pth')

with zipfile.ZipFile('csmsam_results.zip', 'w') as zf:
    for f in files_to_zip:
        if os.path.exists(f):
            zf.write(f)
            print(f'  Added: {f}')

print(f'\nZip created: csmsam_results.zip')

# Download
try:
    from google.colab import files
    files.download('csmsam_results.zip')
except ImportError:
    print('Not in Colab — results saved locally as csmsam_results.zip')

---
## Notes

### This Colab Demo vs Full Experiments

| Aspect | Colab Demo | Full Experiment |
|--------|-----------|------------------|
| Encoder | Lightweight CNN | SAM2.1 Hiera-Large (ViT-H, 307M params) |
| Image size | 512 | 1024 |
| Data | Synthetic | HNTS-MRG 2024 (150 patients) |
| Epochs | 30 | 200 |
| Runtime | ~10 min T4 | ~12h A100 |
| Target DSC | Pipeline test | >75 aggDSC |

### Running Full Experiments
```bash
# 1. Clone and setup
git clone https://github.com/YOUR_USERNAME/csm-sam.git
cd csm-sam
bash scripts/setup_env.sh

# 2. Download real data (register at https://hnts-mrg.grand-challenge.org/)
zenodo_get 11829006 -o data/raw/
python data/preprocess.py --input_dir data/raw --output_dir data/processed

# 3. Train
python train.py --config configs/default.yaml

# 4. Evaluate & visualize
python test.py --checkpoint checkpoints/csmsam_default/best.pth
python visualize.py --checkpoint checkpoints/csmsam_default/best.pth --n_samples 10
```

### Reference
- **HNTS-MRG 2024**: Wahid et al., *Head and Neck Tumor Segmentation for MR-Guided Radiotherapy*, arXiv 2024  
- **SAM2**: Ravi et al., *SAM 2: Segment Anything in Images and Videos*, arXiv 2024  
- **MedSAM2**: bowang-lab, *Medical SAM 2*, GitHub 2024  
- **Zenodo data**: https://zenodo.org/record/11829006